## Load Dataframe


In [ ]:
path = "PH2Dataset\PH2 Dataset images"

In [ ]:

import numpy as np
import matplotlib.pyplot as plt 
import pandas as pd
import PIL.Image as Image
import os
import cv2

In [ ]:
base_path = path
N = 50

In [ ]:
def compute_curvature(contour):
    dx  = np.gradient(contour[:, 0].astype(float))
    dy  = np.gradient(contour[:, 1].astype(float))
    ddx = np.gradient(dx)
    ddy = np.gradient(dy)
    numerator   = np.abs(dx * ddy - dy * ddx)
    denominator = (dx**2 + dy**2)**1.5 + 1e-8
    curvature   = numerator / denominator
    return curvature

def curvature_based_sampling(contour, N):
    curvature = compute_curvature(contour)
    weights = curvature + 1e-3
    weights /= weights.sum()
    indices = np.random.choice(len(contour), size=N, replace=False, p=weights)
    indices.sort()
    return contour[indices], indices, curvature

def hybrid_sampling(contour, N, curvature_ratio=0.5):
    """
    N Punkte total:
    - (1 - curvature_ratio) * N gleichmäßig verteilt
    - curvature_ratio * N krümmungsbasiert
    curvature_ratio=0.0 → rein uniform
    curvature_ratio=1.0 → rein curvature
    """
    n_uniform    = int(N * (1 - curvature_ratio))
    n_curvature  = N - n_uniform

    # ── 1. Uniform-Anteil ──
    uniform_pts = uniform_sampling(contour, n_uniform)

    # ── 2. Curvature-Anteil ──
    curvature = compute_curvature(contour)
    weights = curvature + 1e-3
    weights /= weights.sum()

    # bereits durch uniform abgedeckte Indices ausschließen
    uniform_indices = set(
        np.searchsorted(
            np.linspace(0, len(contour), n_uniform, endpoint=False).astype(int),
            np.arange(n_uniform)
        )
    )
    available = np.array([i for i in range(len(contour)) if i not in uniform_indices])
    
    avail_weights = weights[available]
    avail_weights /= avail_weights.sum()

    curv_indices = np.random.choice(available, size=n_curvature, replace=False, p=avail_weights)
    curvature_pts = contour[curv_indices]

    # ── 3. Zusammenführen ──
    combined = np.vstack([uniform_pts, curvature_pts])
    return combined

def uniform_sampling(contour, N):
    contour = contour.astype(float)

    # geschlossene Kontur sicherstellen
    if not np.all(contour[0] == contour[-1]):
        contour = np.vstack([contour, contour[0]])

    # Segmentlängen
    diffs = np.diff(contour, axis=0)
    lengths = np.linalg.norm(diffs, axis=1)

    cumulative = np.cumsum(lengths)
    total_length = cumulative[-1]

    # gleichmäßige Abstände entlang der Länge
    distances = np.linspace(0, total_length, N, endpoint=False)

    sampled_points = []
    for d in distances:
        idx = np.searchsorted(cumulative, d)
        prev_len = cumulative[idx - 1] if idx > 0 else 0

        t = (d - prev_len) / lengths[idx]
        p = (1 - t) * contour[idx] + t * contour[idx + 1]
        sampled_points.append(p)

    return np.array(sampled_points)


In [ ]:
# Testing

for folder in os.listdir(base_path)[:3]:
    folder_path = os.path.join(base_path, folder)

    if os.path.isdir(folder_path):
        bmp_files = []
        for root, _, files in os.walk(folder_path):
            for file in files:
                if file.lower().endswith(".bmp"):
                    bmp_files.append(os.path.join(root, file))

        if len(bmp_files) >= 2:
            img1 = np.array(Image.open(bmp_files[0]).convert("RGB"))
            img2 = np.array(Image.open(bmp_files[1]).convert("L"))

            
            mask = (img2 > 0).astype(np.uint8) * 255
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            contour = max(contours, key=cv2.contourArea).squeeze()
            

            sampled_points, sampled_indices, curvature = curvature_based_sampling(contour, 50)

            sampled_points = hybrid_sampling(contour, 50, curvature_ratio=0.5)
        

            sampled_points = uniform_sampling(contour, N)

            plt.imshow(img1)
            alpha = (img2 > 0).astype(float) * 0.4
            plt.imshow(img2, cmap="Blues", alpha=alpha)
            plt.scatter(sampled_points[:, 0], sampled_points[:, 1],
                        c="red", s=20, zorder=5)
            plt.title(f"{folder} – {len(sampled_points)} krümmungsbasierte Punkte")
            plt.axis("off")

            plt.tight_layout()
            plt.show()

## Calculation of every Image -> List of Points

In [ ]:
def calculate_points(uniform="", size=N, curvature_ratio=0.5, noise_std=0.0):
    """
    noise_std: Standardabweichung des Gaußschen Rauschens in Pixeln (0.0 = kein Rauschen)
    """
    Points_Dict = {}
    Ground_Truth_Dict = {}
    image_Dict = {}

    for folder in os.listdir(base_path):
        folder_path = os.path.join(base_path, folder)

        if os.path.isdir(folder_path):
            bmp_files = []
            for root, _, files in os.walk(folder_path):
                for file in files:
                    if file.lower().endswith(".bmp"):
                        bmp_files.append(os.path.join(root, file))

            if len(bmp_files) >= 2:
                img1 = np.array(Image.open(bmp_files[0]).convert("RGB"))
                img2 = np.array(Image.open(bmp_files[1]).convert("L"))

                mask = (img2 > 0).astype(np.uint8) * 255
                contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
                contour = max(contours, key=cv2.contourArea).squeeze()

                if uniform == "" or uniform == True or uniform == "uniform":
                    sampled_points = uniform_sampling(contour, size)
                elif uniform == "curvature":
                    sampled_points, _, _ = curvature_based_sampling(contour, size)
                elif uniform == "hybrid":
                    sampled_points = hybrid_sampling(contour, size, curvature_ratio=curvature_ratio)

                # ── Rauschen hinzufügen ──
                if noise_std > 0.0:
                    H, W = img1.shape[:2]
                    noise = np.random.normal(0, noise_std, sampled_points.shape)
                    sampled_points = sampled_points + noise
                    # Punkte innerhalb des Bildes halten
                    sampled_points[:, 0] = np.clip(sampled_points[:, 0], 0, W - 1)
                    sampled_points[:, 1] = np.clip(sampled_points[:, 1], 0, H - 1)

                shuffled = sampled_points.copy()
                np.random.shuffle(shuffled)

                Points_Dict[folder] = shuffled
                Ground_Truth_Dict[folder] = contour
                image_Dict[folder] = img1

    return Points_Dict, Ground_Truth_Dict, image_Dict

Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(size=N)

# 1D - 2D:

## Score Estimation

In [ ]:
def points_to_mask(points, shape):
    mask = np.zeros(shape, dtype=np.uint8)
    
    points = np.asarray(points)

    if len(points) < 3:
        return mask

    points = points.astype(np.int32).reshape((-1, 1, 2))

    cv2.fillPoly(mask, [points], 1)
    return mask

In [ ]:
def polygon_iou(points1, points2, image_shape):
    

    mask1 = points_to_mask(points1, image_shape)
    mask2 = points_to_mask(points2, image_shape)

    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()

    iou = intersection / union if union > 0 else 0.0
    return iou, intersection, union

In [ ]:
from scipy.spatial.distance import cdist

def mean_contour_distance(pred_contour: np.ndarray, gt_contour: np.ndarray) -> dict:
    """
    Misst die Abweichung zwischen zwei Konturen in Pixeln.
    
    pred_contour : (N, 2) – deine KDE-Kontur [x, y]
    gt_contour   : (M, 2) – Ground Truth Kontur [x, y]
    
    Returns dict mit mean, median, max (Hausdorff), std
    """
    # Für jeden Pred-Punkt: Abstand zum nächsten GT-Punkt
    dist_matrix = cdist(pred_contour, gt_contour)
    
    pred_to_gt = dist_matrix.min(axis=1)  # (N,) – pred → gt
    gt_to_pred = dist_matrix.min(axis=0)  # (M,) – gt → pred

    # Symmetrisch: beide Richtungen
    all_dists = np.concatenate([pred_to_gt, gt_to_pred])

    return {
        "mean_px":     np.mean(all_dists),
        "median_px":   np.median(all_dists),
        "hausdorff_px": max(pred_to_gt.max(), gt_to_pred.max()),  # worst case
        "std_px":      np.std(all_dists),
    }

In [ ]:
def plot_iou(sampled, ground_truth, image_shape, img_rgb=None, raw_points=None, balloon=False):
    h, w = image_shape

    mask1 = points_to_mask(sampled, (h, w))
    mask2 = points_to_mask(ground_truth, (h, w))

    intersection = np.logical_and(mask1, mask2)
    union        = np.logical_or(mask1, mask2)
    only1        = np.logical_and(mask1, ~mask2)
    only2        = np.logical_and(mask2, ~mask1)

    iou = intersection.sum() / union.sum() if union.sum() > 0 else 0.0

    overlay = np.zeros((h, w, 3), dtype=np.uint8)
    overlay[only1]        = [200, 80,  80]
    overlay[only2]        = [80,  80,  200]
    overlay[intersection] = [100, 200, 100]

    fig, axes = plt.subplots(1, 4, figsize=(24, 6))

    # ── Plot 1: IoU Overlay ──
    if img_rgb is not None:
        axes[0].imshow(img_rgb)
    axes[0].imshow(overlay, alpha=0.5)
    axes[0].set_title(f"IoU Overlay  (IoU = {iou:.4f})")
    axes[0].axis("off")

    from matplotlib.patches import Patch
    legend = [
        Patch(color=[c/255 for c in [100, 200, 100]], label="Intersection"),
        Patch(color=[c/255 for c in [200, 80,  80]],  label="Sampled"),
        Patch(color=[c/255 for c in [80,  80,  200]],  label="Ground Truth"),
    ]
    axes[0].legend(handles=legend, loc="lower left", fontsize=8)

    # ── Plot 2: Ground Truth ──
    gt_overlay = np.zeros((h, w, 3), dtype=np.uint8)
    gt_overlay[mask2.astype(bool)] = [80, 80, 200]
    if img_rgb is not None:
        axes[1].imshow(img_rgb)
    axes[1].imshow(gt_overlay, alpha=0.5)
    axes[1].set_title("Ground Truth")
    axes[1].axis("off")

    # ── Plot 3: Sampled (interpoliert) ──
    s_overlay = np.zeros((h, w, 3), dtype=np.uint8)
    s_overlay[mask1.astype(bool)] = [200, 80, 80]
    if img_rgb is not None:
        axes[2].imshow(img_rgb)
    axes[2].imshow(s_overlay, alpha=0.5)
    axes[2].set_title("Sampled (interpoliert)")
    axes[2].axis("off")

    # ── Plot 4: Rohe Punkte ──
    if img_rgb is not None:
        axes[3].imshow(img_rgb)
    if raw_points is not None:
        raw = np.array(raw_points)
        axes[3].scatter(raw[:, 0], raw[:, 1], c="red", s=15, zorder=5)

    # ── Zentroid nur wenn balloon=True ──
    if balloon and raw_points is not None:
        centroid = np.mean(np.array(raw_points), axis=0)
        axes[3].scatter(*centroid, c="yellow", s=120, marker="*",
                        zorder=7, edgecolors="black", linewidths=0.8, label="Zentroid")
        axes[3].legend(loc="lower left", fontsize=7)

    axes[3].set_title(f"Rohe Punkte (N={len(raw_points)})")
    axes[3].axis("off")

    plt.tight_layout()
    plt.show()




In [ ]:
def plot_iou_comparison(sampled_dict: dict, sorted_points_dict: dict, ground_truth, image_shape, img_rgb=None, raw_points=None):
    """
    sampled_dict:       {"Linear": poly1, "TSP": poly2, ...}
    sorted_points_dict: {"Linear": sorted_raw1, "TSP": sorted_raw2, ...}  ← neu
    """
    h, w = image_shape
    n_methods = len(sampled_dict)
    mask_gt = points_to_mask(ground_truth, (h, w))

    fig, axes = plt.subplots(2, n_methods + 1, figsize=(5 * (n_methods + 1), 10))

    # ── Spalte 0: Ground Truth + Rohe Punkte ──
    gt_overlay = np.zeros((h, w, 3), dtype=np.uint8)
    gt_overlay[mask_gt.astype(bool)] = [80, 80, 200]
    if img_rgb is not None:
        axes[0, 0].imshow(img_rgb)
    axes[0, 0].imshow(gt_overlay, alpha=0.5)
    axes[0, 0].set_title("Ground Truth")
    axes[0, 0].axis("off")

    if img_rgb is not None:
        axes[1, 0].imshow(img_rgb)
    if raw_points is not None:
        raw = np.array(raw_points)
        axes[1, 0].scatter(raw[:, 0], raw[:, 1], c="red", s=15, zorder=5)
    axes[1, 0].set_title(f"Rohe Punkte (N={len(raw_points) if raw_points is not None else 0})")
    axes[1, 0].axis("off")


    # ── Spalten 1..n: je eine Methode ──
    for col, (method_name, sampled) in enumerate(sampled_dict.items(), start=1):
        mask_s = points_to_mask(sampled, (h, w))

        intersection = np.logical_and(mask_s, mask_gt)
        union        = np.logical_or(mask_s, mask_gt)
        only_s       = np.logical_and(mask_s, ~mask_gt)
        only_gt      = np.logical_and(mask_gt, ~mask_s)
        iou = intersection.sum() / union.sum() if union.sum() > 0 else 0.0

        # ── Zeile 0: IoU Overlay ──
        overlay = np.zeros((h, w, 3), dtype=np.uint8)
        overlay[only_s]       = [200, 80,  80]
        overlay[only_gt]      = [80,  80,  200]
        overlay[intersection] = [100, 200, 100]

        if img_rgb is not None:
            axes[0, col].imshow(img_rgb)
        axes[0, col].imshow(overlay, alpha=0.5)
        axes[0, col].set_title(f"{method_name}\nIoU = {iou:.4f}")
        axes[0, col].axis("off")

        from matplotlib.patches import Patch
        legend = [
            Patch(color=[c/255 for c in [100, 200, 100]], label="Intersection"),
            Patch(color=[c/255 for c in [200, 80,  80]],  label="Sampled"),
            Patch(color=[c/255 for c in [80,  80,  200]],  label="Ground Truth"),
        ]
        axes[0, col].legend(handles=legend, loc="lower left", fontsize=7)

        ## ── Zeile 1: Sampled Maske ──
        #s_overlay = np.zeros((h, w, 3), dtype=np.uint8)
        #s_overlay[mask_s.astype(bool)] = [200, 80, 80]
        #if img_rgb is not None:
        #    axes[1, col].imshow(img_rgb)
        #axes[1, col].imshow(s_overlay, alpha=0.5)
        #axes[1, col].set_title(f"{method_name} (Maske)")
        #axes[1, col].axis("off")

        # ── Zeile 2: Punkt-Ordering ──
        ax = axes[1, col]
        if img_rgb is not None:
            ax.imshow(img_rgb)

        if method_name in sorted_points_dict:
            pts = np.array(sorted_points_dict[method_name])
            n   = len(pts)
            cmap = plt.cm.plasma

            # Farbverlauf-Linie zwischen den sortierten Punkten
            for i in range(n):
                p1 = pts[i]
                p2 = pts[(i + 1) % n]
                color = cmap(i / n)
                ax.annotate("", xy=(p2[0], p2[1]), xytext=(p1[0], p1[1]),
                            arrowprops=dict(arrowstyle="->", color=color, lw=1.5))

            # Punkte mit Farbverlauf
            sc = ax.scatter(pts[:, 0], pts[:, 1],
                            c=np.arange(n), cmap="plasma",
                            s=40, zorder=5, edgecolors="white", linewidths=0.5)

            # Nummern nur bei kleinen Punktmengen (sonst zu voll)
            if n <= 30:
                for i, (x, y) in enumerate(pts):
                    ax.text(x + 3, y - 3, str(i), fontsize=6, color="white", zorder=6)
            if "Balloon" in method_name:
                centroid = np.mean(pts, axis=0)
                ax.scatter(*centroid, c="yellow", s=120, marker="*",
                           zorder=7, edgecolors="black", linewidths=0.8, label="Zentroid")
                ax.legend(loc="lower left", fontsize=7)


            plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02, label="Reihenfolge")

        ax.set_title(f"{method_name} (Ordering)")
        ax.axis("off")

    plt.suptitle("Methoden-Vergleich", fontsize=14)
    plt.tight_layout()
    plt.show()

## Interpolations

In [ ]:
def sort_points_by_nearest_neighbor(pts: np.ndarray) -> np.ndarray:
    """Rekonstruiert eine geschlossene Kontur aus unsortierten Punkten via Nearest Neighbor."""
    remaining = list(range(len(pts)))
    order = [remaining.pop(0)]  # Startpunkt beliebig
    
    while remaining:
        last = pts[order[-1]]
        # Nächsten noch nicht besuchten Punkt finden
        distances = np.linalg.norm(pts[remaining] - last, axis=1)
        nearest = remaining[np.argmin(distances)]
        order.append(nearest)
        remaining.remove(nearest)
    
    return pts[order]

### Linear Interpolation


In [ ]:
def LinearInterpolation(points: list, sorting_function: callable, interpolation_factor: int = 10 ) -> np.ndarray:
    pts = np.array(points, dtype=float)
    
    # ── Unsortierte Punkte zuerst ordnen ──
    pts = sorting_function(pts)
    
    interpolated_points = _LinearInterpolation(pts, interpolation_factor)
    
    return np.array(interpolated_points)

def _LinearInterpolation(pts: list, interpolation_factor: int):
    interpolated_points = []
    for i in range(len(pts)):
        p1 = pts[i]
        p2 = pts[(i + 1) % len(pts)]
        for t in np.linspace(0, 1, num=interpolation_factor, endpoint=False):
            interpolated_points.append((1 - t) * p1 + t * p2)
    return np.array(interpolated_points)

In [ ]:
Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(size=200, noise_std = 3.0)

for n in np.arange(len(Points_Dict))[:3]:
    pts   = list(Points_Dict.values())[n]
    gt    = list(Ground_Truth_Dict.values())[n]
    img   = list(image_Dict.values())[n]
    key   = list(Points_Dict.keys())[n]
    #print(f"Processing {key} with {len(pts)} sampled points and GT of length {len(gt)}")
    interp = LinearInterpolation(pts, sort_points_by_nearest_neighbor)
    iou, inter, union = polygon_iou(interp, gt, img.shape[:2])
    print(f"{key}: IoU={iou:.4f}, Intersection={inter} px, Union={union} px")
    plot_iou(interp, gt, img.shape[:2], img_rgb=img, raw_points=pts)

In [ ]:
results = []

for n in np.arange(len(Points_Dict)):
    pts   = list(Points_Dict.values())[n]
    gt    = list(Ground_Truth_Dict.values())[n]
    img   = list(image_Dict.values())[n]
    key   = list(Points_Dict.keys())[n]
    interp = LinearInterpolation(pts, sort_points_by_nearest_neighbor)
    iou, inter, union = polygon_iou(interp, gt, img.shape[:2])
    #print(f"{key}: IoU={iou:.4f}, Intersection={inter} px, Union={union} px")
    results.append(iou)

np.mean(results)

### Ballon Force

In [ ]:
def _BallonSorting(points: list) -> np.ndarray:
    pts = np.array(points, dtype=float)
    centroid = np.mean(pts, axis=0)

    # ── 1. Winkel + Distanz zum Zentroid ──
    angles = np.arctan2(pts[:, 1] - centroid[1],
                        pts[:, 0] - centroid[0])
    distances = np.linalg.norm(pts - centroid, axis=1)

    # Normalisieren auf [0, 1]
    angles_norm    = (angles - angles.min())    / (angles.max()    - angles.min() + 1e-8)
    distances_norm = (distances - distances.min()) / (distances.max() - distances.min() + 1e-8)

    # ── 2. Kombinierter Score (Winkel dominiert, Distanz bricht Ties) ──
    alpha = 0.97  # Gewichtung Winkel vs. Distanz
    score = alpha * angles_norm + (1 - alpha) * distances_norm

    order = np.argsort(score)
    return pts[order]


def BalloonInterpolation(points: list, interpolation_factor: int = 10, balloon_force: float = 0.1, smooth_window: int = 5) -> np.ndarray:
    
    pts = _BallonSorting(points)
    # ── 2. Linear interpolieren ──
    interpolated_points = _LinearInterpolation(pts, interpolation_factor)

    return np.array(interpolated_points)

In [ ]:
Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(size=200, noise_std = 3.0)

# Plot Balloon
for n in np.arange(len(Points_Dict))[:3]:
    pts   = list(Points_Dict.values())[n]
    gt    = list(Ground_Truth_Dict.values())[n]
    img   = list(image_Dict.values())[n]
    key   = list(Points_Dict.keys())[n]
    interp = BalloonInterpolation(pts)
    iou, inter, union = polygon_iou(interp, gt, img.shape[:2])
    print(f"{key}: IoU={iou:.4f}, Intersection={inter} px, Union={union} px")
    plot_iou(interp, gt, img.shape[:2], img_rgb=img, raw_points=pts, balloon=True)


In [ ]:
results = []

for n in np.arange(len(Points_Dict)):
    pts   = list(Points_Dict.values())[n]
    gt    = list(Ground_Truth_Dict.values())[n]
    img   = list(image_Dict.values())[n]
    key   = list(Points_Dict.keys())[n]
    interp = BalloonInterpolation(pts)
    iou, inter, union = polygon_iou(interp, gt, img.shape[:2])
    #print(f"{key}: IoU={iou:.4f}, Intersection={inter} px, Union={union} px")
    results.append(iou)

np.mean(results)

### TSP

In [ ]:
# Not a real complete TSP, as it is not computationally feasible to solve exactly for large point sets, but a heuristic that tries to find a good ordering of the points to form a closed contour.

from scipy.spatial.distance import cdist

def sort_points_tsp(points: np.ndarray) -> np.ndarray:
    n = len(points)
    dist_matrix = cdist(points, points)

    # ── 1. Nearest Neighbor Startlösung ──
    remaining = list(range(n))
    route = [remaining.pop(0)]
    while remaining:
        distances = dist_matrix[route[-1], remaining]
        nearest_idx = np.argmin(distances)
        route.append(remaining.pop(nearest_idx))

    # ── 2. 2-opt ──
    improved = True
    while improved:
        improved = False
        for i in range(1, n - 1):
            for j in range(i + 1, n):
                a, b = route[i - 1], route[i]
                c, d = route[j], route[(j + 1) % n]
                if dist_matrix[a, b] + dist_matrix[c, d] > dist_matrix[a, c] + dist_matrix[b, d] + 1e-10:
                    route[i:j + 1] = route[i:j + 1][::-1]
                    improved = True

    # ── 3. Or-opt ──
    improved = True
    while improved:
        improved = False
        for seg_len in [1, 2, 3]:
            for i in range(n):
                seg_indices = [(i + k) % n for k in range(seg_len)]
                seg = [route[idx] for idx in seg_indices]
                seg_set = set(seg)

                before_seg = route[(i - 1) % n]
                after_seg  = route[(i + seg_len) % n]

                cost_remove = (dist_matrix[before_seg, seg[0]]
                             + dist_matrix[seg[-1], after_seg]
                             - dist_matrix[before_seg, after_seg])

                new_route = [x for x in route if x not in seg_set]
                
                for j in range(len(new_route)):
                    next_j = new_route[(j + 1) % len(new_route)]
                    cost_insert = (dist_matrix[new_route[j], seg[0]]
                                 + dist_matrix[seg[-1], next_j]
                                 - dist_matrix[new_route[j], next_j])

                    if cost_insert < cost_remove - 1e-10:
                        insert_pos = j + 1
                        route = new_route[:insert_pos] + seg + new_route[insert_pos:]
                        improved = True
                        break
                if improved:
                    break
            if improved:
                break

    return points[route]

In [ ]:
# Plot TSP

for n in np.arange(len(Points_Dict))[:3]:
    pts   = list(Points_Dict.values())[n]
    gt    = list(Ground_Truth_Dict.values())[n]
    img   = list(image_Dict.values())[n]
    key   = list(Points_Dict.keys())[n]
    interp = LinearInterpolation(pts, sort_points_tsp)
    iou, inter, union = polygon_iou(interp, gt, img.shape[:2])
    print(f"{key}: IoU={iou:.4f}, Intersection={inter} px, Union={union} px")
    plot_iou(interp, gt, img.shape[:2], img_rgb=img, raw_points=pts)

In [ ]:
Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(size=200, noise_std = 3.0)
results = []
# Plot Balloon
for n in np.arange(len(Points_Dict)):
    pts   = list(Points_Dict.values())[n]
    gt    = list(Ground_Truth_Dict.values())[n]
    img   = list(image_Dict.values())[n]
    key   = list(Points_Dict.keys())[n]
    interp = LinearInterpolation(pts, sort_points_tsp)
    iou, inter, union = polygon_iou(interp, gt, img.shape[:2])
    #print(f"{key}: IoU={iou:.4f}, Intersection={inter} px, Union={union} px")
    #plot_iou(interp, gt, img.shape[:2], img_rgb=img, raw_points=pts, balloon=True)

    results.append(iou)

np.mean(results)


## smooth TSP

In [ ]:
from scipy.interpolate import splprep, splev
from scipy.spatial.distance import cdist

def smooth_contour_spline(sorted_points: np.ndarray, 
                           n_out: int = None,
                           smoothing: float = None,
                           k: int = 3) -> np.ndarray:
    """
    Glättet eine geordnete, geschlossene Punktmenge via parametrischen B-Spline.

    Parameter:
    ----------
    sorted_points : (N, 2) array – TSP-sortierte Punkte
    n_out         : Anzahl Output-Punkte (default = len(sorted_points))
    smoothing     : Glättungsstärke. None → automatisch (scipy-Standard: s = N).
                    Größere Werte = stärker geglättet, kleinere = näher an den Rohdaten.
    k             : Spline-Grad (3 = kubisch, empfohlen)

    Returns:
    --------
    (n_out, 2) array – geglättete, gleichmäßig gesamplete Konturpunkte
    """
    pts = sorted_points.copy()
    n = len(pts)

    if n_out is None:
        n_out = n

    # Kurve schließen: ersten Punkt ans Ende anhängen
    pts_closed = np.vstack([pts, pts[0]])

    x, y = pts_closed[:, 0], pts_closed[:, 1]

    # s = smoothing-Faktor:
    # s=0  → Interpolation (kein Glätten, läuft durch alle Punkte)
    # s=N  → scipy-Standard, gute Balance
    # s>>N → sehr stark geglättet
    s = smoothing if smoothing is not None else n

    try:
        tck, u = splprep([x, y], s=s, k=k, per=True)  # per=True = periodisch/geschlossen
    except Exception:
        # Fallback falls splprep scheitert (z.B. zu wenige Punkte)
        tck, u = splprep([x, y], s=s, k=min(k, n - 1), per=True)

    # Gleichmäßig gesamplete Parameterwerte
    u_new = np.linspace(0, 1, n_out, endpoint=False)
    x_smooth, y_smooth = splev(u_new, tck)

    return np.column_stack([x_smooth, y_smooth])


def tsp_with_smoothing(points: np.ndarray,
                       n_out: int = None,
                       smoothing: float = None) -> np.ndarray:
    """
    Komplette Pipeline: TSP-Sortierung → Spline-Glättung → gleichmäßiges Resampling.
    
    Parameter:
    ----------
    points    : (N, 2) array – ungeordnete, verrauschte Punkte
    n_out     : Anzahl Output-Punkte (default = N)
    smoothing : Spline-Glättung (None = automatisch anhand Rausch-Niveau)

    Returns:
    --------
    (n_out, 2) array – geglättete Kontur
    """
    sorted_pts = sort_points_tsp(points)
    smoothed   = smooth_contour_spline(sorted_pts, n_out=n_out, smoothing=smoothing)
    return smoothed

In [ ]:
for n in np.arange(len(Points_Dict))[:3]:
    pts   = list(Points_Dict.values())[n]
    gt    = list(Ground_Truth_Dict.values())[n]
    img   = list(image_Dict.values())[n]
    key   = list(Points_Dict.keys())[n]

    # Vorher: interp = LinearInterpolation(pts, sort_points_tsp)
    interp = tsp_with_smoothing(pts, n_out=N, smoothing=N * 5)

    iou, inter, union = polygon_iou(interp, gt, img.shape[:2])
    print(f"{key}: IoU={iou:.4f}, Intersection={inter} px, Union={union} px")
    plot_iou(interp, gt, img.shape[:2], img_rgb=img, raw_points=pts)

In [ ]:
Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(size=200, noise_std=3.0)

for n in np.arange(len(Points_Dict))[:3]:
    pts   = list(Points_Dict.values())[n]
    gt    = list(Ground_Truth_Dict.values())[n]
    img   = list(image_Dict.values())[n]
    key   = list(Points_Dict.keys())[n]

    # Vorher: interp = LinearInterpolation(pts, sort_points_tsp)

    N = 200

    interp = tsp_with_smoothing(pts, n_out=N, smoothing=N * 5)

    iou, inter, union = polygon_iou(interp, gt, img.shape[:2])
    print(f"{key}: IoU={iou:.4f}, Intersection={inter} px, Union={union} px")
    plot_iou(interp, gt, img.shape[:2], img_rgb=img, raw_points=pts)

In [ ]:
Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(size=200, noise_std = 3.0)
results = []

for n in np.arange(len(Points_Dict)):
    pts   = list(Points_Dict.values())[n]
    gt    = list(Ground_Truth_Dict.values())[n]
    img   = list(image_Dict.values())[n]
    key   = list(Points_Dict.keys())[n]
    N = 200

    interp = tsp_with_smoothing(pts, n_out=N, smoothing=N * 5)
    iou, inter, union = polygon_iou(interp, gt, img.shape[:2])
    #print(f"{key}: IoU={iou:.4f}, Intersection={inter} px, Union={union} px")
    #plot_iou(interp, gt, img.shape[:2], img_rgb=img, raw_points=pts, balloon=True)

    results.append(iou)

np.mean(results)


In [ ]:
from itertools import product

def grid_search_smoothing(Points_Dict, Ground_Truth_Dict, image_Dict,
                           smoothing_values, n_values, std_values,
                           sizes=[200, 300]):
    """
    Grid-Search über smoothing × n_out × noise_std × size.
    Gibt ein dict mit allen Ergebnissen + beste Konfiguration zurück.
    """
    results = []

    for size, std in product(sizes, std_values):

        # Daten neu berechnen mit aktuellem std und size
        Points_Dict_run, Ground_Truth_Dict_run, image_Dict_run = calculate_points(
            size=size, noise_std=std
        )

        for smoothing, n_out_factor in product(smoothing_values, n_values):

            ious = []

            for key in Points_Dict_run:
                pts = Points_Dict_run[key]
                gt  = Ground_Truth_Dict_run[key]
                img = image_Dict_run[key]

                try:
                    interp = tsp_with_smoothing(pts, n_out=size * n_out_factor,
                                                smoothing=smoothing)
                    iou, _, _ = polygon_iou(interp, gt, img.shape[:2])
                    ious.append(iou)
                except Exception as e:
                    print(f"  Fehler bei {key} (size={size}, std={std}, "
                          f"s={smoothing}, n={n_out_factor}): {e}")

            if ious:
                mean_iou = np.mean(ious)
                results.append({
                    "size":      size,
                    "noise_std": std,
                    "smoothing": smoothing,
                    "n_factor":  n_out_factor,
                    "mean_iou":  mean_iou,
                    "n_images":  len(ious),
                })
                print(f"size={size:3d} | std={std} | s={smoothing:6.1f} | "
                      f"n×={n_out_factor} | IoU={mean_iou:.4f}  (n={len(ious)})")

    # ── Beste Konfiguration ──
    best = max(results, key=lambda r: r["mean_iou"])
    print("\n" + "="*60)
    print("BESTE KONFIGURATION:")
    for k, v in best.items():
        print(f"  {k}: {v}")

    return results, best

In [ ]:
smoothing_values = [0, N*1, N*2, N*5, N*10, N*20, N*50]
n_values         = [1, 5, 10, 20]          # n_out = size * n_factor
std_values       = [0, 3, 5]

#results, best = grid_search_smoothing(
#    Points_Dict, Ground_Truth_Dict, image_Dict,
#    smoothing_values=smoothing_values,
#    n_values=n_values,
#    std_values=std_values,
#    sizes=[200, 300]
#)



## KDE Estimation

In [ ]:
from scipy.ndimage import gaussian_filter
from skimage.measure import find_contours

def kde_contour(points: np.ndarray, 
                img_shape: tuple, 
                sigma: float,
                threshold: float = None) -> np.ndarray:
    H, W = img_shape

    # ── 1. Punkte auf Grid rendern ──
    density = np.zeros((H, W), dtype=np.float32)
    for x, y in points:
        xi, yi = int(round(x)), int(round(y))
        if 0 <= xi < W and 0 <= yi < H:
            density[yi, xi] += 1.0

    # ── 2. Gauß-Glättung ──
    density = gaussian_filter(density, sigma=max(sigma, 1.0))  # min sigma=1 auch bei noise=0
    density /= density.max()

    # ── 3. Threshold: von grob nach fein suchen bis Kontur gefunden ──
    if threshold is not None:
        thresholds = [threshold]
    else:
        nonzero = density[density > 1e-4]
        if len(nonzero) == 0:
            raise ValueError("KDE-Dichtekarte ist leer – keine Punkte im Bild.")
        # Kandidaten von hoch nach niedrig durchprobieren
        thresholds = [np.percentile(nonzero, p) for p in [50, 30, 20, 10, 5, 1]]

    contours = []
    used_threshold = thresholds[0]
    for t in thresholds:
        contours = find_contours(density, level=t)
        if contours:
            used_threshold = t
            break

    if not contours:
        raise ValueError(f"Keine Kontur gefunden. Dichte-Max={density.max():.4f}, "
                         f"probierte Thresholds={thresholds}")

    # Längste Kontur = Hauptkontur
    contour = max(contours, key=len)
    return contour[:, ::-1]  # (row,col) → (x,y)

In [ ]:
noise_std = 3.0

sigmas = [2, 3, 4, 5, 6]
sampling = ["uniform"]


for sampling_method in sampling:
    for sigma in sigmas:
        
        Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(
            size=300, noise_std=noise_std, uniform=sampling_method
        )

        results = []

        for key in Points_Dict:  # sauberer als index-basiert
            pts = Points_Dict[key]
            gt  = Ground_Truth_Dict[key]
            img = image_Dict[key]

            contour_kde = kde_contour(pts, img.shape[:2], sigma=sigma)
            iou, inter, union = polygon_iou(contour_kde, gt, img.shape[:2])
            results.append(iou)
        
        print(f"{sampling_method} | sigma={sigma} | IoU={np.mean(results):.4f}")  # ← results, nicht result

In [ ]:
noise_std = 5.0

sigmas = [2, 3, 4, 5, 6]
sampling = ["uniform", "hybrid"]

for sampling_method in sampling:
    for sigma in sigmas:
        
        Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(
            size=200, noise_std=noise_std, uniform=sampling_method
        )
        
        results = []

        for n in np.arange(len(Points_Dict)):
            pts = list(Points_Dict.values())[n]
            gt  = list(Ground_Truth_Dict.values())[n]
            img = list(image_Dict.values())[n]
            key = list(Points_Dict.keys())[n]

            contour_kde = kde_contour(pts, img.shape[:2], sigma=sigma)
            metrics = mean_contour_distance(contour_kde, gt.astype(float))
            results.append(metrics)
        # Durchschnitt über alle Bilder
        mean_over_imgs = np.mean([r["mean_px"] for r in results])
        print(f"  → Ø mean_px über alle Bilder: {mean_over_imgs:.2f}px\n")

## KDE Flood Fill

In [ ]:
from scipy.ndimage import  binary_fill_holes

def kde_contour_floodfill(points: np.ndarray,
                           img_shape: tuple,
                           sigma: float,
                           threshold: float = None) -> np.ndarray:
    H, W = img_shape

    # ── 1. KDE ──
    density = np.zeros((H, W), dtype=np.float32)
    for x, y in points:
        xi, yi = int(round(x)), int(round(y))
        if 0 <= xi < W and 0 <= yi < H:
            density[yi, xi] += 1.0

    density = gaussian_filter(density, sigma=max(sigma, 1.0))
    density /= density.max()

    # ── 2. Threshold: hoch genug damit Kontur geschlossen ist ──
    if threshold is None:
        nonzero = density[density > 1e-4]
        # Höheres Percentil = dickere/geschlossenere Konturlinie
        for p in [60, 50, 40, 30, 20, 10]:
            t = np.percentile(nonzero, p)
            binary = (density >= t).astype(np.uint8)
            # Prüfen ob Kontur geschlossen: Flood Fill von Ecke aus
            test = binary.copy()
            cv2.floodFill(test, None, seedPoint=(0, 0), newVal=2)
            # Centroid der Punkte
            cx = int(np.mean(points[:, 0]))
            cy = int(np.mean(points[:, 1]))
            if test[cy, cx] != 2:  # Centroid nicht erreichbar von außen = geschlossen
                break
    else:
        binary = (density >= threshold).astype(np.uint8)

    # ── 3. Centroid als sicherer innerer Seed ──
    cx = int(np.mean(points[:, 0]))
    cy = int(np.mean(points[:, 1]))

    # ── 4. Flood Fill von innen (Hintergrund = 0, Konturring = 1) ──
    # Invertieren: Innen = 0, Kontur = 1 → von Centroid füllen
    filled = binary.copy()
    cv2.floodFill(filled, None, seedPoint=(cx, cy), newVal=2)

    # Alles was mit 2 markiert = Inneres + Konturring
    inner_mask = (filled == 2).astype(np.uint8)
    inner_mask = binary_fill_holes(inner_mask).astype(np.uint8)

    # ── 5. Kontur aus Maske extrahieren ──
    contours_cv, _ = cv2.findContours(inner_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours_cv:
        raise ValueError(f"Keine Kontur nach Flood Fill für sigma={sigma}")

    contour_cv = max(contours_cv, key=cv2.contourArea).squeeze()
    return contour_cv  # bereits (N, 2) in x, y

In [ ]:
noise_std = 3.0

Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(size=300, noise_std=noise_std)

sigma = 4

for key in (np.random.choice(list(Points_Dict.keys()), 3)):
    pts = Points_Dict[key]
    gt  = Ground_Truth_Dict[key]
    img = image_Dict[key]

    contour_kde = kde_contour_floodfill(pts, img.shape[:2], sigma=sigma, threshold=0.0005)
    iou, inter, union = polygon_iou(contour_kde, gt, img.shape[:2])
    
    print(f"{key} | sigma={sigma} | IoU={iou:.4f}")
    plot_iou(contour_kde, gt, img.shape[:2], img_rgb=img, raw_points=pts)

## hybrid


In [ ]:
def kde_contour_hybrid(points: np.ndarray,
                        img_shape: tuple,
                        sigma: float,
                        threshold: float = None,
                        expand_iters: int = 3) -> np.ndarray:
    H, W = img_shape

    # ── 1. KDE → geschlossene Maske (wie bisher) ──
    density = np.zeros((H, W), dtype=np.float32)
    for x, y in points:
        xi, yi = int(round(x)), int(round(y))
        if 0 <= xi < W and 0 <= yi < H:
            density[yi, xi] += 1.0

    density = gaussian_filter(density, sigma=max(sigma, 1.0))
    density /= density.max()

    if threshold is None:
        nonzero = density[density > 1e-4]
        for p in [60, 50, 40, 30, 20, 10]:
            t = np.percentile(nonzero, p)
            binary = (density >= t).astype(np.uint8)
            test = binary.copy()
            cv2.floodFill(test, None, seedPoint=(0, 0), newVal=2)
            cx = int(np.mean(points[:, 0]))
            cy = int(np.mean(points[:, 1]))
            if test[cy, cx] != 2:
                break
    else:
        binary = (density >= threshold).astype(np.uint8)

    cx = int(np.mean(points[:, 0]))
    cy = int(np.mean(points[:, 1]))
    filled = binary.copy()
    cv2.floodFill(filled, None, seedPoint=(cx, cy), newVal=2)
    inner_mask = binary_fill_holes(filled == 2).astype(np.uint8)

    # ── 2. Maske iterativ zu den Punkten hin expandieren ──
    # Punktwolke als Ziel-Dichtekarte
    point_mask = np.zeros((H, W), dtype=np.float32)
    for x, y in points:
        xi, yi = int(round(x)), int(round(y))
        if 0 <= xi < W and 0 <= yi < H:
            point_mask[yi, xi] = 1.0
    point_density = gaussian_filter(point_mask, sigma=sigma * 1.5)
    point_density /= point_density.max()

    # Expansion: Maske wächst dort wo Punktdichte hoch ist
    expanded_mask = inner_mask.copy().astype(np.float32)
    for _ in range(expand_iters):
        # Morphologisch dilaten
        dilated = cv2.dilate(expanded_mask, np.ones((3, 3), np.uint8), iterations=1)
        # Nur dort wachsen wo Punktdichte > Schwellwert
        grow_zone = (dilated - expanded_mask).astype(bool)
        grow_zone &= (point_density > 0.1)
        expanded_mask[grow_zone] = 1.0

    expanded_mask = binary_fill_holes(expanded_mask.astype(bool)).astype(np.uint8)

    # ── 3. Kontur extrahieren ──
    contours_cv, _ = cv2.findContours(expanded_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours_cv:
        raise ValueError("Keine Kontur nach Expansion.")

    return max(contours_cv, key=cv2.contourArea).squeeze()

In [ ]:
noise_std = 3.0

Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(size=200, noise_std=noise_std)

sigma = 4

results = []

for key in ((list(Points_Dict.keys()))):
    pts = Points_Dict[key]
    gt  = Ground_Truth_Dict[key]
    img = image_Dict[key]

    contour_kde = kde_contour_hybrid(pts, img.shape[:2], sigma=sigma)
    iou, inter, union = polygon_iou(contour_kde, gt, img.shape[:2])
    
    print(f"{key} | sigma={sigma} | IoU={iou:.4f}")
    results.append(iou)
    print(iou)

np.mean(results)

In [ ]:
noise_std = 3.0

Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(size=200, noise_std=noise_std)

sigma = 4

for key in (np.random.choice(list(Points_Dict.keys()), 3)):
    pts = Points_Dict[key]
    gt  = Ground_Truth_Dict[key]
    img = image_Dict[key]

    contour_kde = kde_contour_hybrid(pts, img.shape[:2], sigma=sigma)
    iou, inter, union = polygon_iou(contour_kde, gt, img.shape[:2])
    
    print(f"{key} | sigma={sigma} | IoU={iou:.4f}")
    plot_iou(contour_kde, gt, img.shape[:2], img_rgb=img, raw_points=pts)

## real hybrid

In [ ]:
from skimage.draw import polygon as sk_polygon

def kde_contour_hybrid(points: np.ndarray,
                        img_shape: tuple,
                        sigma_inner: float,   # größer → sicher geschlossen, aber zu klein
                        sigma_outer: float,   # kleiner → näher an echten Punkten
                        threshold_outer: float = None) -> np.ndarray:
    H, W = img_shape

    # ── 1. Innere Maske via Floodfill (stabil, geschlossen) ──
    density = np.zeros((H, W), dtype=np.float32)
    for x, y in points:
        xi, yi = int(round(x)), int(round(y))
        if 0 <= xi < W and 0 <= yi < H:
            density[yi, xi] += 1.0

    density_inner = gaussian_filter(density, sigma=max(sigma_inner, 1.0))
    density_inner /= density_inner.max()

    nonzero = density_inner[density_inner > 1e-4]
    cx = int(np.mean(points[:, 0]))
    cy = int(np.mean(points[:, 1]))

    for p in [60, 50, 40, 30, 20, 10]:
        t = np.percentile(nonzero, p)
        binary = (density_inner >= t).astype(np.uint8)
        test = binary.copy()
        cv2.floodFill(test, None, seedPoint=(0, 0), newVal=2)
        if test[cy, cx] != 2:
            break

    filled = binary.copy()
    cv2.floodFill(filled, None, seedPoint=(cx, cy), newVal=2)
    inner_mask = binary_fill_holes(filled == 2).astype(np.uint8)

    # ── 2. Äußere Kontur via kde_contour (näher an echten Punkten) ──
    density_outer = gaussian_filter(density, sigma=max(sigma_outer, 1.0))
    density_outer /= density_outer.max()

    if threshold_outer is not None:
        thresholds = [threshold_outer]
    else:
        nonzero_outer = density_outer[density_outer > 1e-4]
        thresholds = [np.percentile(nonzero_outer, p) for p in [10, 5, 2, 1]]

    outer_contours = []
    for t in thresholds:
        outer_contours = find_contours(density_outer, level=t)
        if outer_contours:
            break

    if not outer_contours:
        raise ValueError("Keine äußere Kontur gefunden.")

    outer_contour = max(outer_contours, key=len)  # (N, 2) row, col
    outer_mask = np.zeros((H, W), dtype=np.uint8)
    rr, cc = sk_polygon(outer_contour[:, 0], outer_contour[:, 1], (H, W))
    outer_mask[rr, cc] = 1
    outer_mask = binary_fill_holes(outer_mask).astype(np.uint8)

    # ── 3. Kombination: Schnittmenge innere Sicherheit + äußere Grenze ──
    # Union: alles was inner ODER outer abdeckt, aber begrenzt durch outer
    combined_mask = np.maximum(inner_mask, outer_mask)  # union
    combined_mask = binary_fill_holes(combined_mask).astype(np.uint8)

    # ── 4. Kontur extrahieren ──
    contours_cv, _ = cv2.findContours(combined_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours_cv:
        raise ValueError("Keine finale Kontur.")

    return max(contours_cv, key=cv2.contourArea).squeeze()

In [ ]:
noise_std = 3.0

Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(size=300, noise_std=noise_std)

sigma = 5

for key in (np.random.choice(list(Points_Dict.keys()), 3)):
    pts = Points_Dict[key]
    gt  = Ground_Truth_Dict[key]
    img = image_Dict[key]

    contour_kde = kde_contour_hybrid(
    pts, img.shape[:2],
    sigma_inner=7,   # groß → sicher geschlossen
    sigma_outer=2,   # klein → näher an echten Punkten
)
    iou, inter, union = polygon_iou(contour_kde, gt, img.shape[:2])
    
    print(f"{key} | sigma={sigma} | IoU={iou:.4f}")
    plot_iou(contour_kde, gt, img.shape[:2], img_rgb=img, raw_points=pts)



## Alpha Shape

In [ ]:
import alphashape

def alpha_shape_contour(points: np.ndarray, alpha_factor: float = 0.02) -> np.ndarray:
    from shapely.geometry import MultiPolygon, Polygon, MultiPoint

    # Alpha relativ zur Punktwolken-Ausdehnung skalieren
    extent = max(points[:, 0].max() - points[:, 0].min(),
                 points[:, 1].max() - points[:, 1].min())
    
    # Adaptiv: größtes alpha das noch ein einzelnes Polygon ergibt
    for factor in [alpha_factor * 0.5, alpha_factor, alpha_factor * 2, alpha_factor * 4]:
        alpha = factor / extent  # ← normiert auf Bildgröße
        shape = alphashape.alphashape(points, alpha)
        
        if isinstance(shape, MultiPolygon):
            # Nimm größtes Fragment nur wenn es dominant ist
            geoms = sorted(shape.geoms, key=lambda g: g.area, reverse=True)
            if geoms[0].area / sum(g.area for g in geoms) > 0.8:
                shape = geoms[0]
            else:
                continue  # zu fragmentiert → nächsten factor probieren
        
        if isinstance(shape, Polygon) and not shape.is_empty:
            break
    else:
        shape = MultiPoint(points).convex_hull

    if not isinstance(shape, Polygon) or shape.is_empty:
        shape = MultiPoint(points).convex_hull

    x, y = shape.exterior.xy
    return np.column_stack([x, y])

In [ ]:
noise_std = 3.0

Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(size=200, noise_std=noise_std)

for key in np.random.choice(list(Points_Dict.keys()), 3):
    pts = Points_Dict[key]
    gt  = Ground_Truth_Dict[key]
    img = image_Dict[key]

    contour = alpha_shape_contour(pts, alpha_factor=0.3)
    iou, inter, union = polygon_iou(contour, gt, img.shape[:2])
    
    print(f"{key} | IoU={iou:.4f}")
    plot_iou(contour, gt, img.shape[:2], img_rgb=img, raw_points=pts)

In [ ]:
noise_std = 5.0

Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(size=300, noise_std=noise_std)

for factor in [x * 0.5 + 4 for x in range(10)]:
    results = []
    for key in Points_Dict:
        pts = Points_Dict[key]
        gt  = Ground_Truth_Dict[key]
        img = image_Dict[key]

        try:
            contour = alpha_shape_contour(pts, alpha_factor=factor)
            iou, _, _ = polygon_iou(contour, gt, img.shape[:2])
            results.append(iou)
        except Exception as e:
            print(f"  Fehler {key} alpha={alpha}: {e}")

    print(f"factor={factor:.3f} | Ø IoU={np.mean(results):.4f}  (n={len(results)})")

Best method :=  alpha shape with alpha_factor 5.0 ~ 92.5 % on   N = 200 and std = 3.0;
                factor=8.000 | Ø IoU=0.8858                     N = 300 and std = 5.0

In [ ]:
noise_std = 3.0

Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(size=200, noise_std=noise_std)

for key in np.random.choice(list(Points_Dict.keys()), 5):
    pts = Points_Dict[key]
    gt  = Ground_Truth_Dict[key]
    img = image_Dict[key]

    contour = alpha_shape_contour(pts, alpha_factor=8)
    iou, inter, union = polygon_iou(contour, gt, img.shape[:2])
    
    print(f"{key} | IoU={iou:.4f}")
    plot_iou(contour, gt, img.shape[:2], img_rgb=img, raw_points=pts)

## Plotting Methods

In [ ]:
# Plot all methods

Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(size=50, uniform="") 

for key in range(5):

    n = list(Points_Dict.keys())[key]

    sorted_points_dict = {
    "Linear": sort_points_by_nearest_neighbor(np.array(Points_Dict[n])),
    "TSP":    sort_points_tsp(np.array(Points_Dict[n])),
    "Balloon": _BallonSorting(np.array(Points_Dict[n])),
    }

    sampled_dict = {
            "Linear":      LinearInterpolation(Points_Dict[n], sort_points_by_nearest_neighbor),
            "TSP":         LinearInterpolation(Points_Dict[n], sort_points_tsp),
            "Balloon":     BalloonInterpolation(Points_Dict[n]),
        }

    plot_iou_comparison(
    sampled_dict=sampled_dict,
    sorted_points_dict=sorted_points_dict,
    ground_truth=Ground_Truth_Dict[n],
    image_shape=image_Dict[n].shape[:2],
    img_rgb=image_Dict[n],
    raw_points=Points_Dict[n],
)

### Sampling

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tqdm

point_counts = [20, 50, 100, 150, 200]
balloon_forces = [1]
uniform_sampled = ["", "curvature", "hybrid"]
curvature_ratios = [x * 0.1 for x in range(10)]  # Nur relevant für "hybrid", aber wir testen alle Kombinationen



In [ ]:
from scipy.stats import gaussian_kde
from scipy.ndimage import binary_fill_holes

def points_to_probability_map(points, image_shape, bandwidth=None):
    H, W = image_shape
    pts = np.array(points, dtype=float)

    # Normalisieren auf [0, 1]
    pts_norm = pts.copy()
    pts_norm[:, 0] /= W
    pts_norm[:, 1] /= H

    # Automatische Bandwidth basierend auf Punktanzahl
    if bandwidth is None:
        bandwidth = 1 / np.sqrt(len(pts))

    # Grid über das Bild
    xs = np.linspace(0, 1, W)
    ys = np.linspace(0, 1, H)
    xx, yy = np.meshgrid(xs, ys)
    grid = np.vstack([xx.ravel(), yy.ravel()])

    # KDE
    kde = gaussian_kde(pts_norm.T, bw_method=bandwidth)
    density = kde(grid).reshape(H, W)

    # Normalisieren auf [0, 1]
    density = (density - density.min()) / (density.max() - density.min() + 1e-8)

    # ── Inneres füllen ──
    binary = density > 0.3        # Kontur als binäre Maske
    filled = binary_fill_holes(binary)  # Loch füllen

    # Probability Map: Inneres = 1.0, Kontur = density, Außen = 0
    prob_map = density.copy()
    prob_map[filled] = np.maximum(prob_map[filled], 1.0)

    return prob_map, filled.astype(np.float32)

In [ ]:
Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(size=150, noise_std = 0.0)

fig, axes = plt.subplots(3, 5, figsize=(25, 15))

H, W = image_Dict[n].shape[:2]
gt_mask = points_to_mask(Ground_Truth_Dict[n], (H, W)).astype(bool)

# Ground Truth Overlay vorbereiten
gt_overlay = np.zeros((H, W, 3), dtype=np.uint8)
gt_overlay[gt_mask] = [80, 80, 200]

for col, n_pts in enumerate([50, 100, 150, 200, 300]):
    pts = np.array(Points_Dict[n])[:n_pts]
    prob_map, mask = points_to_probability_map(pts, (H, W))
    mask_bool = mask.astype(bool)

    intersection = np.logical_and(mask_bool, gt_mask)
    union        = np.logical_or(mask_bool, gt_mask)
    iou = intersection.sum() / (union.sum() + 1e-8)

    # ── Zeile 0: Probability Map + Punkte ──
    axes[0, col].imshow(image_Dict[n])
    axes[0, col].imshow(prob_map, alpha=0.6, cmap="hot")
    axes[0, col].scatter(pts[:, 0], pts[:, 1], c="cyan", s=15, zorder=5)
    axes[0, col].set_title(f"N={n_pts} | IoU={iou:.3f}")
    axes[0, col].axis("off")

    # ── Zeile 1: Ground Truth ──
    axes[1, col].imshow(image_Dict[n])
    axes[1, col].imshow(gt_overlay, alpha=0.5)
    axes[1, col].set_title("Ground Truth")
    axes[1, col].axis("off")

    # ── Zeile 2: Intersection Overlay ──
    only_pred  = np.logical_and(mask_bool, ~gt_mask)
    only_gt    = np.logical_and(gt_mask, ~mask_bool)

    overlay = np.zeros((H, W, 3), dtype=np.uint8)
    overlay[only_pred]   = [200, 80,  80]   # rot   = nur Prediction
    overlay[only_gt]     = [80,  80,  200]  # blau  = nur Ground Truth
    overlay[intersection]= [100, 200, 100]  # grün  = Intersection

    axes[2, col].imshow(image_Dict[n])
    axes[2, col].imshow(overlay, alpha=0.5)
    axes[2, col].set_title(f"Intersection (IoU={iou:.3f})")
    axes[2, col].axis("off")

    from matplotlib.patches import Patch
    legend = [
        Patch(color=[c/255 for c in [100, 200, 100]], label="Intersection"),
        Patch(color=[c/255 for c in [200, 80,  80]],  label="Prediction"),
        Patch(color=[c/255 for c in [80,  80,  200]],  label="Ground Truth"),
    ]
    axes[2, col].legend(handles=legend, loc="lower left", fontsize=7)

plt.suptitle("KDE Probability Map – Vergleich nach Punktanzahl", fontsize=14)
plt.tight_layout()
plt.show()

## INR

In [ ]:
import torch
import torch.nn as nn
import numpy as np

class PointSetINR(nn.Module):
    """
    Nimmt eine ungeordnete Menge von Konturpunkten und schätzt für
    beliebige Koordinaten die Wahrscheinlichkeit, zur Kontur zu gehören.
 
    Input:
        points      – [B, N, 2]  Konturpunkte, normalisiert auf [-1, 1]
        query_coords – [B, Q, 2]  Query-Koordinaten, normalisiert auf [-1, 1]
 
    Output:
        [B, Q, 1]  Wahrscheinlichkeit ∈ (0, 1)
    """
 
    def __init__(self, latent_dim: int = 128):
        super().__init__()
 
        # Encoder: jeden Punkt einzeln einbetten
        self.point_encoder = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim),
        )
 
        # Decoder: latent + query_xy + min_dist → Wahrscheinlichkeit
        # Input-Dim: latent_dim + 2 (xy) + 1 (min_dist)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim + 3, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )
 
    def forward(self, points: torch.Tensor, query_coords: torch.Tensor) -> torch.Tensor:
        B, N, _ = points.shape
        Q = query_coords.size(1)
 
        # --- Latent Code (Max + Mean Pooling) ----------------------------
        feats = self.point_encoder(points)              # [B, N, D]
        lat_max, _ = torch.max(feats, dim=1, keepdim=True)   # [B, 1, D]
        lat_mean   = torch.mean(feats, dim=1, keepdim=True)  # [B, 1, D]
        latent     = (lat_max + lat_mean).expand(-1, Q, -1)  # [B, Q, D]
 
        # --- Explizite Geometrie: min. Distanz zum nächsten Input-Punkt --
        # [B, Q, 1, 2] − [B, 1, N, 2] → [B, Q, N, 2]
        diff = query_coords.unsqueeze(2) - points.unsqueeze(1)
        dists = torch.norm(diff, dim=-1)                      # [B, Q, N]
        min_dist = dists.min(dim=-1, keepdim=True).values     # [B, Q, 1]
 
        # --- Alles zusammenführen ----------------------------------------
        combined = torch.cat([latent, query_coords, min_dist], dim=-1)  # [B, Q, D+3]
        return self.decoder(combined)                                    # [B, Q, 1]
 

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss für starkes Klassenungleichgewicht.
    alpha  – Gewicht für positive Klasse (Kontur-Pixel), typisch 0.75–0.9
    gamma  – Fokus-Exponent; höher = mehr Fokus auf schwere Beispiele
    """
 
    def __init__(self, alpha: float = 0.80, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
 
    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        bce  = F.binary_cross_entropy(pred, target, reduction="none")
        pt   = torch.exp(-bce)
        # alpha skaliert positive Klasse hoch, (1-alpha) negative
        at   = self.alpha * target + (1 - self.alpha) * (1 - target)
        focal = at * (1 - pt) ** self.gamma * bce
        return focal.mean()

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
import torch.optim as optim

class ContourINRDataset(Dataset):
    """
    Erzeugt für jedes Bild:
      - normalisierte Konturpunkte als Input
      - zufällig gesampelte Query-Koordinaten
      - Gauß-Heatmap-Labels (0–1) für jede Query
 
    points_dict : {key: np.array[N, 2]}  – Konturpunkte in Pixel-Koordinaten
    gt_dict     : {key: np.array[M, 2]}  – Ground-Truth Kontur (zum Zeichnen)
    img_dict    : {key: np.array[H, W(, C)]}  – Originalbilder (nur für H, W)
    """
 
    def __init__(
        self,
        points_dict: dict,
        gt_dict: dict,
        img_dict: dict,
        num_queries: int = 4096,
        sigma: float = 3.0,
        near_ratio: float = 0.6,
    ):
        self.keys       = list(points_dict.keys())
        self.points     = points_dict
        self.gt         = gt_dict
        self.imgs       = img_dict
        self.num_q      = num_queries
        self.sigma      = sigma        # Breite der Gauß-Glocke in Pixeln
        self.near_ratio = near_ratio   # Anteil der Queries nah an der Kontur
 
    def __len__(self):
        return len(self.keys)
 
    def __getitem__(self, idx):
        key    = self.keys[idx]
        img    = self.imgs[key]
        H, W   = img.shape[:2]
 
        # 1. Input-Punkte normalisieren auf [-1, 1] (x = Spalte, y = Zeile)
        pts = self.points[key].astype(np.float32).copy()
        pts[:, 0] = (pts[:, 0] / W) * 2.0 - 1.0   # x
        pts[:, 1] = (pts[:, 1] / H) * 2.0 - 1.0   # y
        pts_t = torch.from_numpy(pts)
 
        # 2. Heatmap über Distanz-Transformation
        canvas  = np.zeros((H, W), dtype=np.uint8)
        cv2.drawContours(canvas, [self.gt[key].astype(np.int32)], -1, 1, 1)
        inv     = (1 - canvas).astype(np.uint8)
        dist    = cv2.distanceTransform(inv, cv2.DIST_L2, cv2.DIST_MASK_PRECISE)
        heatmap = np.exp(-(dist ** 2) / (2.0 * self.sigma ** 2)).astype(np.float32)
 
        # 3. Sampling: mehr Punkte nah an der Kontur
        near_mask = heatmap > 0.05
        near_idx  = np.argwhere(near_mask)   # [K, 2] (row, col)
        all_idx   = np.argwhere(heatmap >= 0.0)
 
        n_near = int(self.num_q * self.near_ratio)
        n_far  = self.num_q - n_near
 
        sel_near = near_idx[np.random.choice(len(near_idx), n_near, replace=True)]
        sel_far  = all_idx [np.random.choice(len(all_idx),  n_far,  replace=True)]
        coords   = np.vstack([sel_near, sel_far])   # [Q, 2] (row, col)
 
        q_row, q_col = coords[:, 0], coords[:, 1]
 
        # Labels
        labels = torch.from_numpy(heatmap[q_row, q_col]).unsqueeze(-1)  # [Q, 1]
 
        # Query-Koordinaten normalisieren  (GLEICHE Konvention wie Input-Punkte)
        q_x = (q_col.astype(np.float32) / W) * 2.0 - 1.0
        q_y = (q_row.astype(np.float32) / H) * 2.0 - 1.0
        q_coords = torch.from_numpy(np.stack([q_x, q_y], axis=1))  # [Q, 2]
 
        return pts_t, q_coords, labels

In [ ]:
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = PointSetINR(latent_dim=128).to(device)
 
 
def train_inr(
    points_dict: dict,
    gt_dict: dict,
    img_dict: dict,
    epochs: int = 100,
    batch_size: int = 8,
    lr: float = 1e-3,
    num_queries: int = 4096,
    sigma: float = 3.0,
):
    dataset = ContourINRDataset(points_dict, gt_dict, img_dict,
                                num_queries=num_queries, sigma=sigma)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                         num_workers=0, pin_memory=True)
 
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = FocalLoss(alpha=0.80, gamma=2.0)
 
    model.train()
    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        for pts, q_coords, labels in loader:
            pts, q_coords, labels = (
                pts.to(device),
                q_coords.to(device),
                labels.to(device),
            )
            optimizer.zero_grad()
            preds = model(pts, q_coords)
            loss  = criterion(preds, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
 
        scheduler.step()
 
        if epoch % 10 == 0:
            avg = total_loss / len(loader)
            lr_ = scheduler.get_last_lr()[0]
            print(f"Epoch {epoch:3d}/{epochs}  Loss: {avg:.4f}  LR: {lr_:.2e}")
 
    print("Training abgeschlossen.")

In [ ]:
def predict_mask(
    model: nn.Module,
    points: np.ndarray,
    H: int,
    W: int,
    threshold: float = 0.3,
) -> np.ndarray:
    """
    Erzeugt eine Wahrscheinlichkeits-Heatmap [H, W] ∈ [0, 1].
 
    points : np.array[N, 2]  in Pixel-Koordinaten [x, y]
    H, W   : Bilddimensionen
    """
    model.eval()
 
    # Query-Grid  (GLEICHE Konvention wie Dataset: x=col, y=row)
    y_lin = torch.linspace(-1, 1, H)
    x_lin = torch.linspace(-1, 1, W)
    grid_y, grid_x = torch.meshgrid(y_lin, x_lin, indexing="ij")
    coords = torch.stack([grid_x, grid_y], dim=-1).reshape(1, -1, 2).to(device)
 
    # Punkte normalisieren  (GLEICHE Konvention wie Dataset)
    pts = points.astype(np.float32).copy()
    pts[:, 0] = (pts[:, 0] / W) * 2.0 - 1.0   # x
    pts[:, 1] = (pts[:, 1] / H) * 2.0 - 1.0   # y
    pts_t = torch.from_numpy(pts).unsqueeze(0).to(device)   # [1, N, 2]
 
    with torch.no_grad():
        pred = model(pts_t, coords)          # [1, H*W, 1]
 
    heatmap = pred.squeeze().reshape(H, W).cpu().numpy()    # [H, W]
    mask    = (heatmap >= threshold).astype(np.uint8) * 255
 
    return heatmap, mask

In [ ]:
Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(size=200, noise_std = 0.0)

#train_inr(Points_Dict, Ground_Truth_Dict, image_Dict,
#           epochs=100, batch_size=8, lr=1e-3)

ltrain_inr(Points_Dict, Ground_Truth_Dict, image_Dict,
          epochs=100, batch_size=4, lr=1e-3,
          num_queries=1024, sigma=3.0)
 

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.spatial import cKDTree
from skimage.morphology import skeletonize
import cv2


def chamfer_distance(pred_mask: np.ndarray, gt_mask: np.ndarray) -> float:
    pred_pts = np.argwhere(pred_mask)
    gt_pts   = np.argwhere(gt_mask)
    if len(pred_pts) == 0 or len(gt_pts) == 0:
        return float("inf")
    d_p2g, _ = cKDTree(gt_pts).query(pred_pts)
    d_g2p, _ = cKDTree(pred_pts).query(gt_pts)
    return float((d_p2g.mean() + d_g2p.mean()) / 2)


def extract_contour_from_heatmap(prob_map: np.ndarray, threshold: float = 0.3) -> np.ndarray:
    binary = (prob_map > threshold).astype(np.uint8)
    return skeletonize(binary)  # bool array, 1px dünn


for index in np.random.choice(len(list(Points_Dict.keys())), size=5):

    n    = list(Points_Dict.keys())[index]
    H, W = image_Dict[n].shape[:2]

    # GT: 1px Konturlinie
    gt_img = np.zeros((H, W), dtype=np.uint8)
    cv2.drawContours(gt_img, [Ground_Truth_Dict[n].astype(int)], -1, 1, 1)
    gt_mask = gt_img.astype(bool)

    gt_overlay = np.zeros((H, W, 3), dtype=np.uint8)
    gt_overlay[gt_mask] = [80, 80, 200]

    fig, axes = plt.subplots(3, 5, figsize=(25, 15))
    model.eval()
    point_steps = [50, 100, 150, 200, 300]

    for col, n_pts in enumerate(point_steps):
        current_pts = np.array(Points_Dict[n])[:n_pts]

        prob_map, _ = predict_mask(model, current_pts, H, W)

        # Predicted Kontur: skelettiert → 1px dünn, vergleichbar mit GT
        pred_contour = extract_contour_from_heatmap(prob_map, threshold=0.3)

        # Chamfer Distance (in Pixeln) statt IoU
        cd = chamfer_distance(pred_contour, gt_mask)

        # Fehler-Overlay: pred_contour vs. gt_mask
        only_pred    = np.logical_and(pred_contour, ~gt_mask)
        only_gt      = np.logical_and(gt_mask,      ~pred_contour)
        intersection = np.logical_and(pred_contour,  gt_mask)

        overlay = np.zeros((H, W, 3), dtype=np.uint8)
        overlay[only_pred]    = [200, 80,  80 ]
        overlay[only_gt]      = [80,  80,  200]
        overlay[intersection] = [100, 200, 100]

        # ── Zeile 0: Heatmap + Input-Punkte ──
        axes[0, col].imshow(image_Dict[n])
        axes[0, col].imshow(prob_map, alpha=0.6, cmap="magma", vmin=0, vmax=1)
        axes[0, col].scatter(current_pts[:, 0], current_pts[:, 1],
                             c="cyan", s=15, edgecolors="black", zorder=5)
        axes[0, col].set_title(f"INR Heatmap (N={n_pts})")
        axes[0, col].axis("off")

        # ── Zeile 1: Ground Truth Kontur ──
        axes[1, col].imshow(image_Dict[n])
        axes[1, col].imshow(gt_overlay, alpha=0.8)
        axes[1, col].set_title("Ground Truth Kontur")
        axes[1, col].axis("off")

        # ── Zeile 2: Pred-Kontur vs. GT-Kontur ──
        axes[2, col].imshow(image_Dict[n])
        axes[2, col].imshow(overlay, alpha=0.8)
        axes[2, col].set_title(f"Chamfer: {cd:.1f}px")
        axes[2, col].axis("off")

        axes[2, col].legend(handles=[
            Patch(color=[100/255, 200/255, 100/255], label="Match"),
            Patch(color=[200/255, 80/255,  80/255],  label="Over-est."),
            Patch(color=[80/255,  80/255,  200/255], label="Missing"),
        ], loc="lower left", fontsize=8)

    plt.suptitle(f"INR Kontur-Schätzung | Sample: {n}", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

## Gaussian Splatting

Here we will just interpolate through gaussian Splattings. every Point is described as a bell curve, pointing in the direction of PCA next distribution. In this way we can interpolate the Contur.

In [ ]:
import numpy as np
from scipy.spatial import KDTree
from scipy.ndimage import binary_fill_holes

def points_to_probability_map(points, image_shape, k_neighbors=5, splat_scale=0.2):
    """
    Erstellt eine Probability Map mittels lokaler PCA (Gaussian Splats).
    
    :param points: Liste oder Array von [x, y] Koordinaten.
    :param image_shape: (H, W) des Zielbildes.
    :param k_neighbors: Anzahl der Nachbarn für die lokale PCA (min. 3).
    :param splat_scale: Skalierungsfaktor für die Größe der Splats (Standard: 1.0).
    """
    H, W = image_shape
    pts = np.array(points, dtype=float)
    
    # Grid über das Bild erstellen (Shape: H, W, 2)
    xs = np.arange(0, W)
    ys = np.arange(0, H)
    xx, yy = np.meshgrid(xs, ys)
    grid = np.stack([xx, yy], axis=-1)
    
    # Leere Dichte-Map
    density = np.zeros((H, W), dtype=float)
    
    # Zu wenige Punkte für eine PCA? 
    if len(pts) < k_neighbors:
        return density, density

    # KDTree für schnelle Nachbarschaftssuche
    tree = KDTree(pts)
    # k_neighbors + 1, da der Punkt selbst als sein eigener nächster Nachbar gefunden wird
    _, indices = tree.query(pts, k=k_neighbors + 1)
    
    for i, pt in enumerate(pts):
        # Hole die lokalen Nachbarn
        neighbors = pts[indices[i]]
        
        # Kovarianzmatrix (PCA) berechnen. rowvar=False bedeutet: Spalten sind x und y
        cov = np.cov(neighbors, rowvar=False)
        
        # Numerische Stabilität (falls Punkte exakt auf einer Linie liegen oder alle am selben Ort)
        cov += np.eye(2) * 1e-4
        
        # Splat skalieren
        cov *= splat_scale
        
        # Inverse Kovarianzmatrix für die Gauß-Formel
        inv_cov = np.linalg.inv(cov)
        
        # Abstandsvektor vom Grid zum aktuellen Punkt (H, W, 2)
        diff = grid - pt
        
        # Mahalanobis-Distanz quadriert berechnen: (x-p)^T * Sigma^-1 * (x-p)
        # np.einsum ist hier ein extremer Performance-Boost, um Matrix-Multiplikation 
        # über das gesamte Grid Vektor-basiert auszuführen.
        mahalanobis_sq = np.einsum('hwc,cd,hwd->hw', diff, inv_cov, diff)
        
        # Gaußglocke auswerten (Maximalwert an der Punktkoordinate ist exakt 1.0)
        splat = np.exp(-0.5 * mahalanobis_sq)
        
        # Wir nehmen das Maximum der Splats, damit die Wahrscheinlichkeit bei
        # dichten Punktwolken nicht ins Unendliche addiert wird.
        density = np.maximum(density, splat)

  # ── Inneres füllen ──
    # threshold anpassen, je nachdem wie "fett" der Kontur-Übergang sein soll
    prob_map = density.copy()

    # Vektorisierter Vergleich: Erzeugt direkt eine 2D-Boolean-Maske
    binary = prob_map > 0.3        
    
    # Loch füllen
    filled = binary_fill_holes(binary)
    
    return prob_map, filled.astype(np.float32)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import splprep, splev
from skimage.morphology import skeletonize

def fit_spline_to_mask(binary_mask, smooth=10.0, num_points=100):
    """
    Legt einen glatten B-Spline durch eine binäre Maske.
    
    :param binary_mask: 2D Numpy Array (boolean). Sollte idealerweise linienartig sein.
    :param smooth: Glättungsfaktor (s-Parameter). Höher = glatter, ignoriert kleine Ausreißer.
    :param num_points: Anzahl der Punkte, aus denen der finale Spline bestehen soll.
    """
    # 1. Maske skelettieren (auf 1 Pixel Breite reduzieren)
    skeleton = skeletonize(binary_mask)
    
    # 2. Koordinaten der Skelett-Pixel holen. 
    # Achtung: np.argwhere gibt (y, x) zurück!
    coords = np.argwhere(skeleton)
    
    if len(coords) < 4:
        print("Zu wenige Punkte für einen Spline!")
        return None, None
        
    # 3. Punkte sortieren (Nearest-Neighbor Walk)
    # Wir starten bei dem Punkt ganz links (minimaler x-Wert)
    start_idx = np.argmin(coords[:, 1])
    ordered_coords = [coords[start_idx]]
    
    # Alle anderen Punkte in eine Rest-Liste packen
    remaining_coords = np.delete(coords, start_idx, axis=0)
    
    # Laufe immer zum nächsten unbesuchten Nachbarn
    while len(remaining_coords) > 0:
        last_pt = ordered_coords[-1]
        
        # Euklidische Distanz vom letzten Punkt zu allen restlichen berechnen
        dists = np.linalg.norm(remaining_coords - last_pt, axis=1)
        nearest_idx = np.argmin(dists)
        
        ordered_coords.append(remaining_coords[nearest_idx])
        remaining_coords = np.delete(remaining_coords, nearest_idx, axis=0)
        
    ordered_coords = np.array(ordered_coords)
    
    # y und x Arrays für den Spline trennen
    y_ordered = ordered_coords[:, 0]
    x_ordered = ordered_coords[:, 1]
    
    # 4. Spline fitten
    # splprep erwartet eine Liste von Arrays [x, y]
    # k=3 bedeutet kubischer Spline (Standard und meistens am besten)
    # s ist der Glättungsparameter
    tck, u = splprep([x_ordered, y_ordered], s=smooth, k=3)
    
    # 5. Spline auswerten
    # u_new ist ein Array von 0.0 bis 1.0 (entlang der Kurve)
    u_new = np.linspace(0, 1.0, num_points)
    x_spline, y_spline = splev(u_new, tck)
    
    return x_spline, y_spline


# --- ANWENDUNGSBEISPIEL (in deinem Loop) ---
# Angenommen 'mask_bool' kommt aus deiner points_to_probability_map Funktion:



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.ndimage import binary_erosion, distance_transform_edt

def contour_gaussian_score(pred_mask, gt_mask, noise_std):
    """
    Berechnet einen Boundary-Score basierend auf dem Abstand der Konturen.
    Nutzt eine Gauß-Verteilung über die Distanz zur Bewertung.
    """
    # 1. Konturen extrahieren (Rand = Maske XOR Erodierte Maske)
    gt_contour = gt_mask ^ binary_erosion(gt_mask)
    pred_contour = pred_mask ^ binary_erosion(pred_mask)
    
    if not gt_contour.any() or not pred_contour.any():
        return 0.0

    # 2. Distanz-Transformation: Berechnet den Abstand jedes Pixels zur nächsten Kontur.
    # ~ kehrt das Array um, da edt den Abstand zu den 0-Werten misst.
    dt_gt = distance_transform_edt(~gt_contour)
    dt_pred = distance_transform_edt(~pred_contour)
    
    # 3. Precision (Wie gut liegt die Vorhersage auf der GT?)
    # Hole die Abstände zur GT für alle Pixel, die auf der Pred-Kontur liegen
    dists_pred_to_gt = dt_gt[pred_contour]
    precision_scores = np.exp(-0.5 * (dists_pred_to_gt / noise_std)**2)
    precision = np.mean(precision_scores)
    
    # 4. Recall (Wie gut wird die GT von der Vorhersage abgedeckt?)
    # Hole die Abstände zur Pred für alle Pixel, die auf der GT-Kontur liegen
    dists_gt_to_pred = dt_pred[gt_contour]
    recall_scores = np.exp(-0.5 * (dists_gt_to_pred / noise_std)**2)
    recall = np.mean(recall_scores)
    
    # 5. F1-Score (Harmonisches Mittel)
    if precision + recall == 0:
        return 0.0
        
    f1_score = 2 * (precision * recall) / (precision + recall)
    return f1_score


# ── DEIN PLOTTING CODE MIT DEM NEUEN SCORER ──

point_list = [50, 100, 150, 200, 300]

#point_list = [200]
# Falls du nur einen testen willst: point_list = [250]

# Dynamische Breite je nach Anzahl der Punkte anpassen
fig, axes = plt.subplots(3, len(point_list), figsize=(5 * len(point_list), 15), squeeze=False)

# Wir nehmen an, dass du 'n' (Index des Bildes) vorher definiert hast
n = list(image_Dict.keys())[0]
H, W = image_Dict[n].shape[:2]
gt_mask = points_to_mask(Ground_Truth_Dict[n], (H, W)).astype(bool)

gt_overlay = np.zeros((H, W, 3), dtype=np.uint8)
gt_overlay[gt_mask] = [80, 80, 200]

noise_std_value = 5.0 # Dein Noise-Level

for col, n_pts in enumerate(point_list):
    Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(size=n_pts, noise_std=noise_std_value, uniform=True)
    pts = np.array(Points_Dict[n])[:n_pts]
    prob_map, mask = points_to_probability_map(pts, (H, W), k_neighbors=3)
    mask_bool = mask.astype(bool)

    # Berechne den neuen Contour Score
    c_score = contour_gaussian_score(mask_bool, gt_mask, noise_std=noise_std_value)

    # ── Zeile 0: Probability Map + Punkte ──
    axes[0, col].imshow(image_Dict[n])
    axes[0, col].imshow(prob_map, alpha=0.6, cmap="hot")
    axes[0, col].scatter(pts[:, 0], pts[:, 1], c="cyan", s=15, zorder=5)
    axes[0, col].set_title(f"N={n_pts} | C-Score={c_score:.3f}")
    axes[0, col].axis("off")

    # ── Zeile 1: Ground Truth ──
    axes[1, col].imshow(image_Dict[n])
    axes[1, col].imshow(gt_overlay, alpha=0.5)
    axes[1, col].set_title("Ground Truth")
    axes[1, col].axis("off")

    # ── Zeile 2: Intersection Overlay ──
    intersection = np.logical_and(mask_bool, gt_mask)
    only_pred  = np.logical_and(mask_bool, ~gt_mask)
    only_gt    = np.logical_and(gt_mask, ~mask_bool)

    overlay = np.zeros((H, W, 3), dtype=np.uint8)
    overlay[only_pred]   = [200, 80,  80]   # rot   = nur Prediction
    overlay[only_gt]     = [80,  80,  200]  # blau  = nur Ground Truth
    overlay[intersection]= [100, 200, 100]  # grün  = Intersection

    axes[2, col].imshow(image_Dict[n])
    axes[2, col].imshow(overlay, alpha=0.5)
    
    # Optional: Du kannst dir auch weiterhin den IoU mit ausgeben lassen zum Vergleich
    iou = intersection.sum() / (np.logical_or(mask_bool, gt_mask).sum() + 1e-8)
    axes[2, col].set_title(f"IoU={iou:.3f} | C-Score={c_score:.3f}")
    axes[2, col].axis("off")

    legend = [
        Patch(color=[c/255 for c in [100, 200, 100]], label="Intersection"),
        Patch(color=[c/255 for c in [200, 80,  80]],  label="Prediction"),
        Patch(color=[c/255 for c in [80,  80,  200]],  label="Ground Truth"),
    ]
    axes[2, col].legend(handles=legend, loc="lower left", fontsize=7)

plt.suptitle("KDE Probability Map – Vergleich (Contour Gaussian Score)", fontsize=14)
plt.tight_layout()
plt.show()

## Delaunay Concave Hull

In [ ]:
from scipy.spatial import Delaunay
from shapely.geometry import MultiPolygon, Polygon, MultiPoint
from shapely.ops import unary_union
import numpy as np

def delaunay_concave_hull(points: np.ndarray, edge_length_quantile: float = 0.85) -> np.ndarray:
    tri = Delaunay(points)
    
    edge_count = {}
    for simplex in tri.simplices:
        for i in range(3):
            edge = tuple(sorted([simplex[i], simplex[(i+1) % 3]]))
            edge_count[edge] = edge_count.get(edge, 0) + 1
    
    boundary_edges = {e for e, c in edge_count.items() if c == 1}
    
    # ── Neu: Threshold relativ zur mittleren Punktdichte ──────────────────────
    # Erwarteter Abstand bei gleichmäßiger Verteilung auf Perimeter
    perimeter_estimate = (
    np.sqrt(
        np.ptp(points[:, 0]) ** 2 +
        np.ptp(points[:, 1]) ** 2
    ) * 2
)
    expected_spacing = perimeter_estimate / len(points)
    
    edge_lengths = {
        e: np.linalg.norm(points[e[0]] - points[e[1]]) for e in boundary_edges
    }
    
    # Quantil nur über Boundary-Kanten, aber gedeckelt auf k * expected_spacing
    q_threshold  = np.quantile(list(edge_lengths.values()), edge_length_quantile)
    density_cap  = expected_spacing * 4.0   # max 4x erwarteter Abstand
    threshold    = min(q_threshold, density_cap)
    # ──────────────────────────────────────────────────────────────────────────
    
    valid_simplices = []
    for simplex in tri.simplices:
        edges    = [tuple(sorted([simplex[i], simplex[(i+1) % 3]])) for i in range(3)]
        boundary = [e for e in edges if e in boundary_edges]
        if all(edge_lengths[e] <= threshold for e in boundary):
            valid_simplices.append(simplex)
    
    if not valid_simplices:
        return np.column_stack(MultiPoint(points).convex_hull.exterior.xy)
    
    polys  = [Polygon(points[s]) for s in valid_simplices if Polygon(points[s]).is_valid]
    merged = unary_union(polys)
    
    if isinstance(merged, MultiPolygon):
        geoms  = sorted(merged.geoms, key=lambda g: g.area, reverse=True)
        merged = geoms[0]
    
    x, y = merged.exterior.xy
    return np.column_stack([x, y])

## alpha shape adaption

In [ ]:
def snap_contour_to_points(contour: np.ndarray, points: np.ndarray, 
                             snap_radius: float = None) -> np.ndarray:
    """
    Zieht jeden Konturpunkt zum nächsten Input-Punkt,
    wenn dieser nahe genug ist. Macht Alpha Shape präziser.
    """
    from scipy.spatial import cKDTree
    
    if snap_radius is None:
        # Automatisch: mittlerer Abstand zwischen Nachbarpunkten
        tree = cKDTree(points)
        dists, _ = tree.query(points, k=2)
        snap_radius = np.median(dists[:, 1]) * 1.5
    
    tree = cKDTree(points)
    snapped = contour.copy().astype(float)
    
    for i, pt in enumerate(contour):
        dist, idx = tree.query(pt)
        if dist < snap_radius:
            snapped[i] = points[idx]
    
    return snapped.astype(np.float32)

# Und im Alpha-Shape direkt einbauen:
def alpha_shape_snapped(points: np.ndarray, alpha_factor: float = 0.015) -> np.ndarray:
    contour = alpha_shape_contour(points, alpha_factor)
    return snap_contour_to_points(contour, points)

## Ensemble-Vote

In [ ]:
def ensemble_contour(points: np.ndarray, img_shape: tuple, sigma: float = 1.5) -> np.ndarray:
    from shapely.geometry import Polygon
    import cv2
    
    H, W = img_shape
    masks = []
    
    # Methode A: Delaunay
    try:
        c = delaunay_concave_hull(points)
        p = Polygon(c)
        if p.is_valid and not p.is_empty:
            m = np.zeros((H, W), np.uint8)
            cv2.fillPoly(m, [c.astype(np.int32)], 1)
            masks.append(m)
    except Exception:
        pass
    
    # Methode B: Alpha Shape (tight)
    try:
        c = alpha_shape_contour(points, alpha_factor=0.01)
        p = Polygon(c)
        if p.is_valid and not p.is_empty:
            m = np.zeros((H, W), np.uint8)
            cv2.fillPoly(m, [c.astype(np.int32)], 1)
            masks.append(m)
    except Exception:
        pass
    
    # Methode C: KDE (deflated – sigma klein, hoher Threshold)
    try:
        c = kde_contour_hybrid(points, img_shape, sigma=max(sigma, 1.0),
                                threshold=None, expand_iters=1)
        m = np.zeros((H, W), np.uint8)
        cv2.fillPoly(m, [c.astype(np.int32)], 1)
        # Maske leicht erodieren um KDE-Overshoot zu kompensieren
        m = cv2.erode(m, np.ones((3,3), np.uint8), iterations=2)
        masks.append(m)
    except Exception:
        pass
    
    if not masks:
        raise ValueError("Alle Methoden fehlgeschlagen")
    
    # Majority vote
    vote = np.sum(masks, axis=0)
    threshold = max(1, len(masks) // 2 + 1)  # >50% Mehrheit
    final_mask = (vote >= threshold).astype(np.uint8)
    
    contours_cv, _ = cv2.findContours(final_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    return max(contours_cv, key=cv2.contourArea).squeeze()

In [ ]:
noise_std = 3.0

Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(size=200, noise_std=noise_std)

methods = {
    "Alpha Shape":        lambda pts: alpha_shape_contour(pts, alpha_factor=0.3),
    "Delaunay Hull":      lambda pts: delaunay_concave_hull(pts, edge_length_quantile=0.85),
    "Alpha + Snap":       lambda pts: alpha_shape_snapped(pts, alpha_factor=10),
}

keys = np.random.choice(list(Points_Dict.keys()), 3, replace=False)

fig, axes = plt.subplots(len(keys), len(methods), figsize=(5 * len(methods), 4 * len(keys)))

for row, key in enumerate(keys):
    pts = Points_Dict[key]
    gt  = Ground_Truth_Dict[key]
    img = image_Dict[key]

    for col, (method_name, method_fn) in enumerate(methods.items()):
        ax = axes[row, col]

        try:
            contour = method_fn(pts)
            iou, _, _ = polygon_iou(contour, gt, img.shape[:2])
            title = f"{method_name}\nIoU={iou:.4f}"
            color = "lime"
        except Exception as e:
            contour = None
            title = f"{method_name}\nFEHLER: {e}"
            color = "red"

        ax.imshow(img)
        ax.scatter(pts[:, 0], pts[:, 1], s=6, c="cyan", alpha=0.6, zorder=3)

        # Ground Truth
        if gt is not None and len(gt):
            gt_closed = np.vstack([gt, gt[0]])
            ax.plot(gt_closed[:, 0], gt_closed[:, 1], "w--", linewidth=1.2, alpha=0.7, label="GT")

        # Predicted contour
        if contour is not None and len(contour):
            c_closed = np.vstack([contour, contour[0]])
            ax.plot(c_closed[:, 0], c_closed[:, 1], color=color, linewidth=1.5, label="Pred")

        if row == 0:
            ax.set_title(title, fontsize=10, pad=6)
        else:
            ax.set_title(title, fontsize=10)

        ax.axis("off")

        if col == 0:
            ax.set_ylabel(str(key), fontsize=9)

plt.suptitle(f"Method Comparison  |  N=200, std={noise_std}", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
noise_std = 3.0

Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(size=200, noise_std=noise_std)

results = []
results_delauney = []

for key in list(Points_Dict.keys())[:3]:
    pts = Points_Dict[key]
    gt  = Ground_Truth_Dict[key]
    img = image_Dict[key]

    contour_base    = delaunay_concave_hull(pts, edge_length_quantile=0.90)
    contour_snapped = snap_contour_to_points(contour_base, pts, snap_radius=1)
    
    iou, inter, union = polygon_iou(contour_snapped, gt, img.shape[:2])
    plot_iou(contour_snapped, gt, img.shape[:2], img_rgb=img, raw_points=pts)

for key in list(Points_Dict.keys()):
    pts = Points_Dict[key]
    gt  = Ground_Truth_Dict[key]
    img = image_Dict[key]

    contour_base    = delaunay_concave_hull(pts, edge_length_quantile=0.90)
    contour_snapped = snap_contour_to_points(contour_base, pts, snap_radius=1)
    
    iou, _, _ = polygon_iou(contour_snapped, gt, img.shape[:2])
    iou_delauney, _, _ = polygon_iou(contour_snapped, gt, img.shape[:2])
    results.append(iou)
    results_delauney.append(iou_delauney)

print(f"Mean IoU: {np.mean(results):.4f}")
print(f"Mean IoU: {np.mean(results_delauney):.4f}")

In [ ]:
noise_std = 3.0

Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(size=200, noise_std=noise_std)

for key in np.random.choice(list(Points_Dict.keys()), 5):
    pts = Points_Dict[key]
    gt  = Ground_Truth_Dict[key]
    img = image_Dict[key]

    contour = alpha_shape_contour(pts, alpha_factor=8)
    iou, inter, union = polygon_iou(contour, gt, img.shape[:2])
    
    print(f"{key} | IoU={iou:.4f}")
    plot_iou(contour, gt, img.shape[:2], img_rgb=img, raw_points=pts)

## BALLON ???

In [ ]:
import numpy as np
from scipy.interpolate import splprep, splev

def SmoothSplineInterpolation(points: list, target_points: int = 200, smoothing: float = 2.0) -> np.ndarray:
    pts = _BallonSorting(points)
    
    # splprep erwartet die x- und y-Koordinaten separat (shape: 2xN)
    # per=1 signalisiert, dass es sich um eine geschlossene Kurve handelt (verbindet Ende mit Anfang)
    # s (smoothing) regelt, wie eng die Kurve an den Originalpunkten anliegen muss (0 = exakt)
    tck, u = splprep([pts[:, 0], pts[:, 1]], s=smoothing, per=1)
    
    # Neue, gleichmäßig verteilte Punkte auf der glatten Kurve berechnen
    u_new = np.linspace(0, 1.0, target_points)
    x_smooth, y_smooth = splev(u_new, tck)
    
    return np.column_stack((x_smooth, y_smooth))

In [ ]:
import numpy as np

def BalloonInterpolation_2(points: list, interpolation_factor: int = 10, smooth_window: int = 33) -> np.ndarray:
    """Main pipeline: Sorts points radially, interpolates linearly, and applies smoothing."""
    pts = _BallonSorting(points)
    
    # 1. Linearly interpolate between the sorted points to increase resolution
    interpolated_points = _LinearInterpolation(pts, interpolation_factor)
    interpolated_array = np.array(interpolated_points)
    
    # 2. Apply moving average smoothing to the high-resolution points
    # A larger window size (e.g., 33) is needed here due to the multiplied point density
    smoothed_points = _SmoothContour(interpolated_array, window_size=smooth_window)
    
    return smoothed_points

def _BallonSorting(points: list) -> np.ndarray:
    """Sorts unordered contour points radially around their center of mass."""
    pts = np.array(points, dtype=float)
    centroid = np.mean(pts, axis=0)

    # ── 1. Calculate angle and distance relative to the centroid ──
    angles = np.arctan2(pts[:, 1] - centroid[1],
                        pts[:, 0] - centroid[0])
    distances = np.linalg.norm(pts - centroid, axis=1)

    # Normalize angles and distances to the [0, 1] range to ensure consistent weighting
    angles_norm    = (angles - angles.min())    / (angles.max()    - angles.min() + 1e-8)
    distances_norm = (distances - distances.min()) / (distances.max() - distances.min() + 1e-8)

    # ── 2. Calculate combined score ──
    # The angle primarily dictates the circular ordering.
    # The distance acts as a tie-breaker for points with similar angles.
    alpha = 0.97  # Weighting factor: 97% angle, 3% distance
    score = alpha * angles_norm + (1 - alpha) * distances_norm

    # Sort the points based on the computed score
    order = np.argsort(score)
    return pts[order]

def _LinearInterpolation(pts: list, interpolation_factor: int):
    """Inserts additional points linearly between existing contour points."""
    interpolated_points = []
    
    for i in range(len(pts)):
        p1 = pts[i]
        # Wrap around to the first point to close the contour loop
        p2 = pts[(i + 1) % len(pts)]
        
        # Generate 'interpolation_factor' number of intermediate points
        for t in np.linspace(0, 1, num=interpolation_factor, endpoint=False):
            interpolated_points.append((1 - t) * p1 + t * p2)
            
    return np.array(interpolated_points)

def _SmoothContour(pts: np.ndarray, window_size: int = 5) -> np.ndarray:
    """Smooths a closed contour using a moving average filter."""
    if window_size < 3:
        return pts
        
    # For a moving average on a closed shape, we need to wrap the ends 
    # around (padding) to ensure a seamless transition at the start/end point.
    pad = window_size // 2
    pts_padded = np.vstack((pts[-pad:], pts, pts[:pad]))
    
    # Create a 1D uniform filter kernel for the moving average
    kernel = np.ones(window_size) / window_size
    
    # Apply convolution to smooth X and Y coordinates independently
    x_smooth = np.convolve(pts_padded[:, 0], kernel, mode='valid')
    y_smooth = np.convolve(pts_padded[:, 1], kernel, mode='valid')
    
    return np.column_stack((x_smooth, y_smooth))

In [ ]:
results = []

for n in np.arange(len(Points_Dict)):
    pts   = list(Points_Dict.values())[n]
    gt    = list(Ground_Truth_Dict.values())[n]
    img   = list(image_Dict.values())[n]
    key   = list(Points_Dict.keys())[n]
    interp = SmoothSplineInterpolation(pts, target_points = 200, smoothing = 5)
    iou, inter, union = polygon_iou(interp, gt, img.shape[:2])
    #print(f"{key}: IoU={iou:.4f}, Intersection={inter} px, Union={union} px")
    results.append(iou)

np.mean(results)

In [ ]:
Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(size=200, noise_std = 3.0)

# Plot Balloon
for n in np.arange(len(Points_Dict))[:3]:
    pts   = list(Points_Dict.values())[n]
    gt    = list(Ground_Truth_Dict.values())[n]
    img   = list(image_Dict.values())[n]
    key   = list(Points_Dict.keys())[n]
    interp = SmoothSplineInterpolation(pts, target_points = 400, smoothing = 5)
    iou, inter, union = polygon_iou(interp, gt, img.shape[:2])
    print(f"{key}: IoU={iou:.4f}, Intersection={inter} px, Union={union} px")
    plot_iou(interp, gt, img.shape[:2], img_rgb=img, raw_points=pts, balloon=True)


In [ ]:
results = []

for n in np.arange(len(Points_Dict)):
    pts   = list(Points_Dict.values())[n]
    gt    = list(Ground_Truth_Dict.values())[n]
    img   = list(image_Dict.values())[n]
    key   = list(Points_Dict.keys())[n]
    interp = BalloonInterpolation_2(pts, interpolation_factor = 10, smooth_window = 33)
    iou, inter, union = polygon_iou(interp, gt, img.shape[:2])
    #print(f"{key}: IoU={iou:.4f}, Intersection={inter} px, Union={union} px")
    results.append(iou)

np.mean(results)

In [ ]:
Points_Dict, Ground_Truth_Dict, image_Dict = calculate_points(size=200, noise_std = 3.0)

# Plot Balloon
for n in np.arange(len(Points_Dict))[:3]:
    pts   = list(Points_Dict.values())[n]
    gt    = list(Ground_Truth_Dict.values())[n]
    img   = list(image_Dict.values())[n]
    key   = list(Points_Dict.keys())[n]
    interp = BalloonInterpolation_2(pts, interpolation_factor = 10, smooth_window = 33)
    iou, inter, union = polygon_iou(interp, gt, img.shape[:2])
    print(f"{key}: IoU={iou:.4f}, Intersection={inter} px, Union={union} px")
    plot_iou(interp, gt, img.shape[:2], img_rgb=img, raw_points=pts, balloon=True)
